# Experiment 1 — Phase B analysis

Confirmatory / paper-table-producing analysis of the Phase B eval. Spec:
[`exp1_phase_b_analysis_design.md`](exp1_phase_b_analysis_design.md). Reuses
loader + classifiers + CIs from [`_eval_loader.py`](_eval_loader.py).

Phase A's [`exp1_eval_analysis.ipynb`](exp1_eval_analysis.ipynb) is the per-eval
diagnostic notebook. This notebook produces:
1. The headline numbers for the paper (§2)
2. The MFS-baseline-cleared finding (§3)
3. Per-sense P/R/F1 + confusion (§4)
4. Quote-level error analysis (§5) — the workhorse for going back through quotes
5. The Phase A → Phase B replication panel (§6)
6. Pooled descriptive analyses on the combined 38-lemma corpus (§7)
7. The dev-set appendix table (§8)
8. Methodology notes for the writeup (§9)

## Parameters

In [ ]:
# Registry names — see the `run` column in llm_predictions/*.csv. The notebook never holds
# raw eval IDs; eval_id() resolves and raises if a slot is still null.
PHASE_B = 'phase_b_plain'           # required
PHASE_A = 'phase_a_plain_restricted'           # optional — enables §6 replication + §7 pooled + §8 dev appendix
PHASE_B_NO_CONTEXT = None           # 'phase_b_no_context' once registered (v2)
PHASE_B_JSON = None                 # 'phase_b_json' once registered (deferred per methodology log 2026-05-16c)
CONTAMINATION_PROBE_CSV = None      # per-lemma contamination probe — placeholder (v2)

BOOTSTRAP_N = 10_000
RNG_SEED = 20260516

In [ ]:
import datetime as _dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _eval_loader import (
    load_eval, load_evals, eval_id, EVALS, infer_stage,
    wald_ci, bootstrap_ci_binary, mcnemar, holm_bonferroni,
    AXES, AXIS_VALUES, RATES, PHASE_A_LEMMAS,
    mfs_baseline, kappa_vs_gold, kappa_labels, parse_predicted_number,
    per_quote_outcome_matrix, inspect_quote, quotes_where, inspect_quotes,
    quotes_table, quote_details, quote_picker,
    exp1_candidate_senses, exp1_prediction_breakdown,
    plot_quote_confusion,
    exp1_top_confusion,
    exp1_best_providers, exp1_attach_predicted_sid,
    exp1_modal_votes, exp1_agreement_metrics,
    exp1_label_map, sid_hierarchy_relation,
    compute_lemma_features,
    load_quote_features,
)

# Resolve registry names — clear error if unfilled.
EVAL_ID_PHASE_B = eval_id(PHASE_B)
meta_b, df_b = load_eval(EVAL_ID_PHASE_B)
RNG = np.random.default_rng(RNG_SEED)

# Optional Phase A — gracefully None if not requested.
if PHASE_A:
    EVAL_ID_PHASE_A = eval_id(PHASE_A)
    meta_a, df_a = load_eval(EVAL_ID_PHASE_A)
else:
    EVAL_ID_PHASE_A = None
    meta_a, df_a = None, None

print(f'Phase B: {EVAL_ID_PHASE_B}  ({len(df_b):,} rows, {df_b.provider_label.nunique()} providers, {df_b.template.nunique()} templates)')
if df_a is not None:
    print(f'Phase A: {EVAL_ID_PHASE_A}  ({len(df_a):,} rows)')

## §1. What this eval did

Scope check + Phase-A-leakage warning. If any Phase A lemma sneaks into
this eval, the headline numbers in §2 would mix selection-contaminated
with clean data — fail loudly so the analyst notices.

In [ ]:
def scope_summary(meta, df, label='eval'):
    ts = _dt.datetime.fromtimestamp(meta['created_at_ms'] / 1000).isoformat(timespec='seconds')
    lemmas = sorted(df.lemma.unique())
    quotes_per_lemma = df.groupby('lemma')['quote_idx'].nunique()
    print(f'=== {label} ===')
    print(f'  id              : {meta["id"]}')
    print(f'  created         : {ts}')
    print(f'  rows            : {len(df):,}')
    print(f'  inferred stage  : {infer_stage(set(lemmas))}')
    print(f'  lemmas ({len(lemmas)})    : {lemmas}')
    print(f'  quotes/lemma    : min={quotes_per_lemma.min()}  median={int(quotes_per_lemma.median())}  max={quotes_per_lemma.max()}')
    print(f'  providers ({df.provider_label.nunique()})  : {sorted(df.provider_label.unique())}')
    print(f'  templates ({df.template.nunique()})  : {sorted(df.template.unique())}')
    for a in AXES:
        vs = sorted(df[a].dropna().unique().tolist())
        if len(vs) > 1 or vs:
            print(f'  axis {a:<14}: {vs}')
    print(f'  total spend     : ${df.cost.sum():.4f}')
    print(f'  outcomes        : {df.outcome.value_counts().to_dict()}')

scope_summary(meta_b, df_b, label='Phase B')

# Phase-A-leakage check.
leaked = set(df_b.lemma.unique()) & PHASE_A_LEMMAS
if leaked:
    print(f'\n!!! WARNING: {len(leaked)} Phase A lemma(s) present in Phase B eval: {sorted(leaked)}')
    print('    → headline numbers in §2 would mix selection-contaminated with clean data.')
    print('    → either filter df_b to disjoint lemmas only, or verify the run config.')
else:
    print('\n  ✓ no Phase A lemma leakage — disjoint sets, clean evaluation.')

## §2. Headline table — *the paper table*

Per (provider, config) on Phase B disjoint lemmas: accuracy ± bootstrap CI,
Cohen's κ vs gold, Δ vs MFS baseline, format-failure rate, cost. Ranked by
accuracy. Δ-MFS flags configs that don't clear MFS "convincingly" per
PLAN.md §2.5 — anything below MFS is not a finding.

Outputs a markdown-paste-able DataFrame for direct inclusion in the paper draft.

In [ ]:
# Derive config id (template × axis cell). For Phase B at l1/meta=F the
# axis cell collapses to just def_language.
df_b = df_b.assign(config=df_b.apply(
    lambda r: f"{r.template}|def={r.def_language}", axis=1))
if 'predicted' not in df_b.columns or df_b['predicted'].isna().all():
    df_b['predicted'] = df_b['predicted_number'] if 'predicted_number' in df_b.columns else df_b['output'].map(parse_predicted_number)

# MFS baseline — per-lemma, macro across lemmas present in Phase B only.
mfs_per_lemma, _ = mfs_baseline()
mfs_macro_phaseB = float(np.mean([mfs_per_lemma[l] for l in df_b.lemma.unique() if l in mfs_per_lemma]))
print(f'MFS baseline (macro across {df_b.lemma.nunique()} Phase B lemmas): {100*mfs_macro_phaseB:.2f}%')

In [ ]:
def headline_table(df, mfs_macro):
    rows = []
    for (prov, cfg), g in df.groupby(['provider_label', 'config']):
        n = len(g); ok = int(g.success.sum())
        ff = int((g.outcome == 'format_failure').sum())
        lo, hi = bootstrap_ci_binary(ok, n, n_boot=BOOTSTRAP_N, rng=RNG)
        kap = kappa_vs_gold(g['predicted'], g['gold_number'])
        acc = ok / n
        rows.append({
            'provider': prov, 'config': cfg, 'n': n,
            'acc%': round(100 * acc, 2),
            'ci_lo%': round(100 * lo, 2),
            'ci_hi%': round(100 * hi, 2),
            'kappa': round(kap, 3),
            'delta_mfs_pp': round(100 * (acc - mfs_macro), 1),
            'fmt_fail%': round(100 * ff / n, 2),
            'spent$': round(g.cost.sum(), 4),
            '$/correct': round(g.cost.sum() / ok, 6) if ok else float('inf'),
        })
    t = pd.DataFrame(rows).sort_values('acc%', ascending=False).reset_index(drop=True)
    t['clears_mfs'] = t['ci_lo%'] > 100 * mfs_macro
    return t

headline = headline_table(df_b, mfs_macro_phaseB)
headline

In [ ]:
# Forest plot: ranked (provider, config) cells with 95% bootstrap CI;
# MFS baseline as a vertical reference line.
fig, ax = plt.subplots(figsize=(9, max(4, 0.22 * len(headline))))
y = np.arange(len(headline))
colors = ['C2' if x else 'C3' for x in headline['clears_mfs']]
ax.errorbar(headline['acc%'], y,
            xerr=[headline['acc%'] - headline['ci_lo%'], headline['ci_hi%'] - headline['acc%']],
            fmt='o', ecolor='gray', markerfacecolor='none', capsize=2)
for yi, c in zip(y, colors):
    ax.plot(headline['acc%'].iloc[yi], yi, 'o', color=c, ms=6)
ax.axvline(100*mfs_macro_phaseB, color='black', linestyle='--', alpha=0.5,
           label=f'MFS macro = {100*mfs_macro_phaseB:.1f}%')
ax.set_yticks(y); ax.set_yticklabels(headline['provider'] + ' | ' + headline['config'], fontsize=7)
ax.invert_yaxis()
ax.set_xlabel('Accuracy % (95% bootstrap CI)')
ax.set_title(f'Phase B headline — {meta_b["id"]}')
ax.grid(axis='x', alpha=0.3); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

n_clear = int(headline['clears_mfs'].sum())
print(f'\n{n_clear} of {len(headline)} (provider, config) cells clear MFS (CI lower bound > MFS macro).')
print('Cells below MFS — methodologically: "not a finding" per PLAN.md §2.5:')
print(headline[~headline['clears_mfs']][['provider','config','acc%','delta_mfs_pp']].to_string(index=False))

### Compact alternatives to the 80-row forest plot

The forest plot above lists all 80 (provider, config) cells as individual
rows — accurate, but too tall to use as a paper figure. Two compact
alternatives, both keyed on `headline` (nothing new to compute):

1. **Per-provider spread** — one row per provider (10, not 80), configs
   jittered within the row and coloured by `def_language`, best config per
   provider starred. Same visual language as Experiment 2's `fig_perconfig`
   (`paper_figures_plotly.ipynb`), minus the human-ceiling line — Experiment 1
   has no IAA to draw one from.
2. **Provider × config heatmap** — all 80 cells in a 10×8 grid, colour =
   accuracy%, diverging around the MFS floor so MFS-clearing cells read green
   and sub-MFS cells read red at a glance. Same `RdYlGn`/`center=` convention
   already used for the covariate-sensitivity heatmap in §7.


In [ ]:
# Export helper — mirrors paper_figures_plotly.ipynb's export() convention,
# matplotlib version (this notebook has no plotly dependency).
import os
from pathlib import Path
FIG_OUT = Path(os.environ.get('DIALEXPOL_OUTPUT_DIR', Path.cwd() / 'outputs')) / 'figures'
FIG_OUT.mkdir(parents=True, exist_ok=True)

def export_fig(fig, stem, dpi=200):
    for ext in ('png', 'pdf', 'svg'):
        fig.savefig(FIG_OUT / f'{stem}.{ext}', dpi=dpi, bbox_inches='tight')
    print(f'exported {stem}.{{png,pdf,svg}} -> {FIG_OUT}')


In [ ]:
# Compact per-config plot: one row per provider (not per cell), configs
# jittered within the row and coloured by def_language; best config per
# provider starred; MFS floor as the only reference line (no human-ceiling
# line — Experiment 1 has no IAA to draw one from, unlike fig_perconfig).
prov_order = (headline.groupby('provider')['acc%'].max()
              .sort_values(ascending=False).index.tolist())
n = len(prov_order)
PL, LA, BEST = '#4c72b0', '#dd8452', '#1f2f52'

fig, ax = plt.subplots(figsize=(9, 0.55 * n + 1))
for pi, prov in enumerate(prov_order):
    band = n - 1 - pi  # best provider (pi=0) drawn at the top
    sub = headline[headline['provider'] == prov].sort_values('acc%')
    best_acc = sub['acc%'].max()
    offs = np.linspace(-0.30, 0.30, len(sub))
    for off, (_, row) in zip(offs, sub.iterrows()):
        is_best = row['acc%'] == best_acc
        color = BEST if is_best else (PL if row['config'].endswith('def=pl') else LA)
        ax.errorbar(row['acc%'], band + off,
                    xerr=[[row['acc%'] - row['ci_lo%']], [row['ci_hi%'] - row['acc%']]],
                    fmt='*' if is_best else 'o', color=color,
                    ms=13 if is_best else 6, ecolor='lightgray', capsize=2,
                    zorder=3 if is_best else 2)

ax.axvline(100 * mfs_macro_phaseB, color='black', linestyle='--', alpha=0.5)
ax.set_yticks([n - 1 - i for i in range(n)])
ax.set_yticklabels(prov_order, fontsize=8)
ax.set_ylim(-0.6, n - 0.4)
ax.set_xlabel('Accuracy % (95% bootstrap CI; 8 configs/provider, jittered)')
ax.set_title(f'Phase B — per-provider configuration spread — {meta_b["id"]}')
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=PL, label='def=pl', ms=8),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=LA, label='def=la', ms=8),
    plt.Line2D([0], [0], marker='*', color='w', markerfacecolor=BEST, label='best per provider', ms=14),
    plt.Line2D([0], [0], color='black', linestyle='--', alpha=0.5,
               label=f'MFS macro = {100*mfs_macro_phaseB:.1f}%'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=8)
ax.grid(axis='x', alpha=0.3)
export_fig(fig, 'fig_phaseb_perconfig')
plt.tight_layout(); plt.show()


In [ ]:
# Same 80 cells as a provider x config heatmap — colour = accuracy%,
# diverging around the MFS floor (RdYlGn, center=MFS) so the 34/80
# MFS-clearing split reads directly off the colour rather than a count.
pivot = headline.pivot(index='provider', columns='config', values='acc%').loc[prov_order]

fig, ax = plt.subplots(figsize=(10, 0.5 * len(pivot) + 2))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=100 * mfs_macro_phaseB,
            cbar_kws={'label': 'accuracy %'}, ax=ax, linewidths=0.4, linecolor='white')
ax.set_title(f'Phase B — accuracy by (provider, config) — '
             f'MFS macro = {100*mfs_macro_phaseB:.1f}% — {meta_b["id"]}')
ax.set_xlabel(''); ax.set_ylabel('')
export_fig(fig, 'fig_phaseb_heatmap')
plt.tight_layout(); plt.show()


## §3. Baselines

PLAN.md §2.5 prescribes three baselines for Phase B:

- **MFS** — Most-Frequent-Sense per lemma. Computed in §2 from the gold distribution.
- **No-context LLM** — best Phase B config with the quotation stripped. Placeholder until eval exists.
- **Latin BERT** (Bamman & Burns 2020) — embedding + linear probe. Placeholder until pipeline exists.

In [ ]:
# MFS per Phase B lemma + macro mean — surface which lemmas have a high MFS
# floor (Zipfian distributions where MFS is hard to beat).
phaseB_mfs = pd.DataFrame([
    {'lemma': l, 'mfs_acc%': round(100 * mfs_per_lemma[l], 1)}
    for l in sorted(df_b.lemma.unique()) if l in mfs_per_lemma
]).sort_values('mfs_acc%', ascending=False).reset_index(drop=True)
print(f'MFS macro across Phase B lemmas: {100*mfs_macro_phaseB:.2f}%')
print(f'Per-lemma MFS distribution (high = hard to beat):')
phaseB_mfs

In [ ]:
if PHASE_B_NO_CONTEXT is None:
    print('No-context baseline: not yet run. No released phase_b_no_context run is available')
    print('  (re-run Phase B best config with quotation_wsd field replaced).')
else:
    print(f'No-context baseline eval: {eval_id(PHASE_B_NO_CONTEXT)} — TODO: integrate per-provider comparison.')

print('\nLatin BERT baseline: separate pipeline (Bamman & Burns 2020 embedding + linear probe).')
print('  Tracked outside this notebook; cite when available.')

## §4. Per-sense P/R/F1 + confusion matrices

Per PLAN.md §2.3. Computed per (provider × config × lemma); rendered only
for the headline config and the top-3 worst / top-3 best lemmas (full grid
is too many subplots to scan).

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

def per_sense_prf1(g):
    p = pd.to_numeric(g['predicted'], errors='coerce')
    y = pd.to_numeric(g['gold_number'], errors='coerce')
    mask = p.notna() & y.notna()
    if mask.sum() == 0:
        return None
    labels = sorted(set(y[mask].astype(int)) | set(p[mask].astype(int)))
    prec, rec, f1, sup = precision_recall_fscore_support(
        y[mask].astype(int), p[mask].astype(int), labels=labels, zero_division=0)
    return pd.DataFrame({'sense': labels, 'P': prec, 'R': rec, 'F1': f1, 'n_gold': sup})

# Macro F1 column added to the headline (per the design doc note).
macro_f1 = {}
for (prov, cfg), g in df_b.groupby(['provider_label', 'config']):
    p = per_sense_prf1(g)
    macro_f1[(prov, cfg)] = float(p['F1'].mean()) if p is not None else float('nan')
headline['macro_f1'] = [round(macro_f1.get((r.provider, r.config), float('nan')), 3) for r in headline.itertuples()]
headline.head(10)

In [ ]:
# Per-lemma confusion matrix for the headline config (top-acc cell).
top = headline.iloc[0]
headline_cfg = (top['provider'], top['config'])
print(f'Confusion matrices for headline cell: {headline_cfg}')

# Pick top-3 worst and top-3 best lemmas under this config
per_lemma_acc = (df_b[(df_b.provider_label == top['provider']) & (df_b.config == top['config'])]
                 .groupby('lemma')['success'].mean().sort_values())
lemmas_to_plot = list(per_lemma_acc.head(3).index) + list(per_lemma_acc.tail(3).index)

fig, axs = plt.subplots(2, 3, figsize=(13, 7))
for ax, lemma in zip(axs.flat, lemmas_to_plot):
    sub = df_b[(df_b.provider_label == top['provider'])
               & (df_b.config == top['config'])
               & (df_b.lemma == lemma)]
    p = pd.to_numeric(sub['predicted'], errors='coerce')
    y = pd.to_numeric(sub['gold_number'], errors='coerce')
    mask = p.notna() & y.notna()
    if mask.sum() == 0:
        ax.set_title(f'{lemma} (no data)'); continue
    labels = sorted(set(y[mask].astype(int)) | set(p[mask].astype(int)))
    cm = confusion_matrix(y[mask].astype(int), p[mask].astype(int), labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
                xticklabels=labels, yticklabels=labels, ax=ax, cbar=False)
    ax.set_title(f'{lemma} ({100*sub.success.mean():.1f}%)')
    ax.set_xlabel('predicted'); ax.set_ylabel('gold')
plt.suptitle(f'Top-3 worst (top row) + top-3 best (bottom row) lemmas — {top["provider"]} | {top["config"]}')
plt.tight_layout(); plt.show()

## §5. Error analysis & failure typology

The drill-down section, restructured for fast navigation:

- **§5a** — per-quote outcome matrix (one row per quote, consensus signals)
- **§5b** — covariate effects (Polish gloss, sc-note, genre, length, century)
  - §5b.1 effect-size ranking — biggest signals at the top
  - §5b.2 per-covariate detail panels
- **§5c** — drill-down: one quote (`inspect_quote`)
- **§5d** — drill-down: a filtered cohort (`quotes_where` + `inspect_quotes`)
- **§5e** — top sense confusion pairs (gold → predicted)
- **§5f** — best-model agreement on hard quotes + presentation-position bias
- **§5g** — dictionary-hierarchy distance of wrong picks (near-miss vs different major sense)

### §5a. Per-quote outcome matrix

Wide DataFrame: one row per quote, one column per (provider, config) cell
plus consensus signals (`n_correct`, `n_wrong`, `n_ff`, `difficulty` ∈ [0, 1])
and the joined per-quote features (Polish gloss, genre, century, …).

In [ ]:
# §5a — Per-quote outcome matrix, joined to the per-quote features CSV.
features = load_quote_features()
per_q = per_quote_outcome_matrix(df_b, features_df=features)
print(f'{len(per_q)} unique quotes; columns: {list(per_q.columns)[:15]}…')

# Hardest quotes (failed by ≥80% of providers/configs)
hard = per_q[per_q['difficulty'] >= 0.8].sort_values('difficulty', ascending=False)
print(f'\n{len(hard)} quotes with difficulty ≥ 0.8 (≥80% of cells got them wrong):')
print(hard[['quote_text', 'n_correct', 'n_cells', 'difficulty', 'genre']].head(10).to_string())

In [ ]:
# Consensus signals — split / consensus-fail / consensus-correct
consensus_fail = per_q[per_q['n_correct'] == 0]
consensus_ok = per_q[per_q['n_correct'] == per_q['n_cells']]
split = per_q[(per_q['n_correct'] > 0.3 * per_q['n_cells']) & (per_q['n_correct'] < 0.7 * per_q['n_cells'])]
print(f'consensus failures (all cells wrong)   : {len(consensus_fail)}')
print(f'consensus correct  (all cells right)   : {len(consensus_ok)}')
print(f'split decisions    (30-70% correct)    : {len(split)}')
print(f'\nTop-10 consensus-failure quotes — these are the most informative for error analysis:')
print(consensus_fail.sort_values('n_cells', ascending=False)[['quote_text','n_cells','genre','length_bin']].head(10).to_string())

### §5b. Covariate effects

For each binary/categorical covariate, marginal accuracy and Δ vs the
overall mean. The first two cells give scannable visuals; the third
drops the raw table for reference. Covariates surfaced:

- `has_polish_gloss` — Polish chars or `"…"` / `„…"` gloss in the quote
- `has_sc_note` — editorial `(sc. …)` clarification (stripped from the
  LLM prompt post-2026-05-16d; remaining effect = selection + helper-loss)
- `genre` — source genre (curated via sigla CSV)
- `length_bin` — short / medium / long (<5 / 5-15 / >15 words)
- `century` — 12-17 (derived from `a. NNNN` date field)

In [ ]:
# §5b prep: compute the covariate slice DataFrame `cov` once;
# the next cells render it as bar chart + detail panels.
#
# Quote-level covariates come from the features CSV. Lemma-level covariates
# (n_candidate_senses, quotes_per_sense) are computed from the inventory +
# eval, then binned for the bar-chart treatment. quotes_per_sense is a
# semantic-complexity proxy — low = polysemous + sparse evidence (harder).

lemma_feats = compute_lemma_features(df_b)
def _bin_qps(x):
    if pd.isna(x): return None
    return 'low (<1.5)' if x < 1.5 else 'med (1.5-3)' if x < 3 else 'high (≥3)'
def _bin_ns(x):
    if pd.isna(x): return None
    return 'few (≤5)' if x <= 5 else 'med (6-9)' if x <= 9 else 'many (≥10)'
def _bin_nw(x):
    if pd.isna(x): return None
    return 'very_short (<5)' if x < 5 else 'short (5-9)' if x < 10 else 'med (10-19)' if x < 20 else 'long (≥20)'

lemma_feats['quotes_per_sense_bin'] = lemma_feats['quotes_per_sense'].map(_bin_qps)
lemma_feats['n_senses_bin'] = lemma_feats['n_candidate_senses'].map(_bin_ns)
per_q = per_q.join(lemma_feats[['n_candidate_senses', 'quotes_per_sense',
                                'quotes_per_sense_bin', 'n_senses_bin']], on='lemma')
per_q['n_words_bin'] = per_q['n_words'].map(_bin_nw)

def _slice_acc(per_q, covariate, value=None):
    if value is None:
        return per_q['n_correct'].sum() / per_q['n_cells'].sum() if len(per_q) else float('nan')
    sub = per_q[per_q[covariate] == value]
    if sub.empty:
        return float('nan')
    return sub['n_correct'].sum() / sub['n_cells'].sum()

overall = _slice_acc(per_q, None)
_cov_rows = []
for col in ['has_polish_gloss', 'has_sc_note', 'genre', 'length_bin', 'n_words_bin',
            'century', 'n_senses_bin', 'quotes_per_sense_bin']:
    if col not in per_q.columns:
        continue
    for val in sorted(per_q[col].dropna().unique(), key=str):
        sub = per_q[per_q[col] == val]
        if sub.empty: continue
        n_q = len(sub); n_cells = int(sub['n_cells'].sum()); n_ok = int(sub['n_correct'].sum())
        acc = n_ok / n_cells
        _cov_rows.append({
            'covariate': col, 'value': str(val),
            'n_quotes': n_q, 'n_cells': n_cells,
            'acc%': round(100*acc, 1),
            'delta_overall_pp': round(100*(acc - overall), 1),
        })
cov = pd.DataFrame(_cov_rows)
print(f'Overall accuracy across all cells: {100*overall:.2f}%  (denominator for Δ)')
print(f'{len(cov)} covariate-value slices computed across {cov.covariate.nunique()} covariates.')

In [ ]:
# §5b.1 — Effect-size ranking. Sorted bar chart of |Δ vs overall|.
# Green = above average, red = below. Use this to identify which covariate
# values move the needle before diving into details.
cov_ranked = cov.assign(abs_delta=cov['delta_overall_pp'].abs()).sort_values('abs_delta', ascending=False)
labels = [f"{r.covariate}={r.value}  (n={r.n_quotes})" for _, r in cov_ranked.iterrows()]
deltas = cov_ranked['delta_overall_pp'].values
colors = ['C2' if d > 0 else 'C3' for d in deltas]
fig, ax = plt.subplots(figsize=(9, max(3, 0.30 * len(cov_ranked))))
y = np.arange(len(cov_ranked))
ax.barh(y, deltas, color=colors)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Δ vs overall accuracy (pp)')
ax.set_title('Covariate effect ranking — biggest signals at the top')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()
# Top-5 largest absolute effects (text follow-up — easy to copy into writeup notes)
print('Top 5 |Δ| signals:')
for _, r in cov_ranked.head(5).iterrows():
    sign = '+' if r['delta_overall_pp'] > 0 else ''
    print(f"  {r.covariate}={r.value:<24} n={r.n_quotes:>4}  acc={r['acc%']:>5.1f}%  ({sign}{r.delta_overall_pp} pp vs overall)")

In [ ]:
# §5b.2 — Per-covariate detail panels. One panel per covariate; bars are
# absolute acc% with the dashed line marking the overall mean for context.
cov_names = cov['covariate'].unique()
n = len(cov_names)
fig, axs = plt.subplots(1, n, figsize=(3.2 * n, 4))
if n == 1: axs = [axs]
for ax, c in zip(axs, cov_names):
    sub = cov[cov['covariate'] == c]
    pos = range(len(sub))
    cols = ['C2' if d > 0 else 'C3' for d in sub['delta_overall_pp']]
    ax.barh(pos, sub['acc%'], color=cols)
    ax.set_yticks(pos); ax.set_yticklabels(sub['value'], fontsize=8)
    ax.invert_yaxis()
    ax.axvline(100*overall, color='black', linestyle='--', alpha=0.5)
    ax.set_title(c, fontsize=10); ax.set_xlabel('acc%')
plt.tight_layout(); plt.show()
print(f'(dashed line = overall accuracy = {100*overall:.1f}%)')
print()
print('Raw covariate table:')
cov

#### §5b.3. Per-provider × covariate sensitivity

For each covariate, a heatmap with providers on the y-axis and covariate
values on the x-axis. Cell colour = accuracy on that slice. Reveals which
*providers* are disproportionately affected by which covariate — e.g.
"does any provider survive the sc-note slice?" or "which providers handle
high-polysemy lemmas best?" The §5b.1/§5b.2 charts hide this — they're
pooled across providers.

In [ ]:
# §5b.3 — Per-provider × covariate heatmap. Join the per-quote covariates
# back into df_b so we can group by (provider, covariate-value).
_quote_covs = per_q[['has_polish_gloss', 'has_sc_note', 'genre', 'length_bin',
                     'quotes_per_sense_bin', 'n_senses_bin']].reset_index()
df_b_cov = df_b.merge(_quote_covs, on=['lemma', 'quote_idx'], how='left')

covariates_to_show = ['has_sc_note', 'has_polish_gloss', 'genre', 'quotes_per_sense_bin']
ncov = len(covariates_to_show)
fig, axs = plt.subplots(1, ncov, figsize=(4.2 * ncov, max(4, 0.4 * df_b.provider_label.nunique())))
if ncov == 1: axs = [axs]
for ax, c in zip(axs, covariates_to_show):
    if c not in df_b_cov.columns: continue
    pivot = (df_b_cov.groupby(['provider_label', c])['success'].mean().unstack() * 100).round(1)
    # sort providers by overall mean for consistency across panels
    pivot = pivot.reindex(index=pivot.mean(axis=1).sort_values(ascending=False).index)
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label':'acc%'},
                ax=ax, vmin=0, vmax=80)
    ax.set_title(c); ax.set_ylabel('' if ax is not axs[0] else 'provider')
    ax.set_xlabel(''); ax.tick_params(axis='x', rotation=30)
plt.suptitle('Provider sensitivity per covariate (cell = acc%)', y=1.02)
plt.tight_layout(); plt.show()

#### §5b.4. Cross-cuts — covariate × axis interactions

Three paper-relevant interactions:

- **`has_polish_gloss × def_language`**: does the Polish-gloss benefit
  generalise across both definition languages? (Intuition: gloss might
  help more for `def=la` quotes where the Polish word resolves a
  semantic ambiguity the Latin alone leaves open.)
- **`has_sc_note × task_type`**: does any template handle sc-flagged
  quotes better? (Useful mitigation hint — if `justified` or `decomposed`
  recovers some of the 12 pp loss, the helper effect may be partly
  template-recoverable.)
- **`genre × provider`**: per-provider specialisation by source genre —
  which provider is best at chronicles, legal, glosses, etc.

In [ ]:
# §5b.4 — Three cross-cut heatmaps. All on the same df_b_cov computed in §5b.3.
fig, axs = plt.subplots(1, 3, figsize=(16, 4 + 0.3 * df_b.provider_label.nunique()))

# (a) has_polish_gloss × def_language
pivot_a = (df_b_cov.groupby(['has_polish_gloss', 'def_language'])['success'].mean().unstack() * 100).round(1)
sns.heatmap(pivot_a, annot=True, fmt='.1f', cmap='RdYlGn', center=100*overall, ax=axs[0],
            cbar_kws={'label':'acc%'})
axs[0].set_title('has_polish_gloss × def_language')

# (b) has_sc_note × task_type (= template-without-yaml)
pivot_b = (df_b_cov.groupby(['has_sc_note', 'task_type'])['success'].mean().unstack() * 100).round(1)
sns.heatmap(pivot_b, annot=True, fmt='.1f', cmap='RdYlGn', center=100*overall, ax=axs[1],
            cbar_kws={'label':'acc%'})
axs[1].set_title('has_sc_note × task_type')

# (c) genre × provider — bigger; render separately so it has room
plt.tight_layout(); plt.show()

# genre × provider as its own figure
pivot_c = (df_b_cov.groupby(['genre', 'provider_label'])['success'].mean().unstack() * 100).round(1)
# Order providers by overall acc desc; order genres by mean across providers desc
pivot_c = pivot_c.reindex(columns=pivot_c.mean(axis=0).sort_values(ascending=False).index)
pivot_c = pivot_c.reindex(index=pivot_c.mean(axis=1).sort_values(ascending=False).index)
fig, ax = plt.subplots(figsize=(max(8, 0.7 * pivot_c.shape[1]), max(4, 0.45 * pivot_c.shape[0])))
sns.heatmap(pivot_c, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label':'acc%'}, ax=ax,
            vmin=0, vmax=80)
ax.set_title('genre × provider — acc% (rows sorted by mean across providers)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

### §5c. Browse hard quotes (tabular view)

A scrollable DataFrame of the hardest 50 quotes — sortable by clicking
column headers in Jupyter. Use it to identify quotes of interest, then
either pick one in §5d's dropdown (if `ipywidgets` is installed) or
call `quote_details(df_b, lemma, quote_idx, features_df=features)` to
see the per-(provider, config) breakdown for that quote.

In [ ]:
# §5c — Tabular browse view (replaces print-based inspect_quote).
# DataFrames render as scrollable HTML in Jupyter; click column headers to sort.
from IPython.display import display
print(f'Top 50 hardest quotes (highest fraction of cells wrong).')
print('Sort by clicking any column header; copy a (lemma, quote_idx) into §5d for drill-in.')
print()
quotes_table(per_q, top_n=50, sort_by='difficulty')

### §5d. Drill-down: one quote (separate cells, sortable tables)

Split into independent cells so each table is a native (sortable) DataFrame and
nothing can stack/duplicate the way a live widget did. **Workflow:** the *select*
cell prints a shortlist of the worst-predicted quotes (same data as the §5c table)
and defaults to the hardest one; set `(lemma, qidx)` to any pair from it, run the
cell, then run the cells below. Re-run them after changing the quote.

The cells render, for that one quote:

1. **select + context** — worst-quote shortlist; sets `lemma, qidx`, resolves
   `ctx` / `preds` / `cand`, prints the gold sense and the marked quote;
2. **candidate-sense legend** — the numbered menu the model saw for *this* quote
   (numbering is shuffled per quote; ★ = gold). Decodes the `predicted` numbers;
3. **per-(provider, config) predictions** — each cell's pick with its `pred_sid`
   + Latin/Polish glosses (`def_la` / `def_pl`);
4. **provider × sense heatmap** — vote distribution for this occurrence, gold
   column outlined (`by='task_type'` for prompt-style confusion instead);
5. **predicted-sense breakdown** — one row per predicted sense (glosses + gold
   flag + total) with a count-by-provider pivot: what the gold is confused with.

`quote_picker(...)` (the old single live-updating widget) still exists in
`_eval_loader` if you want it, but it can't give sortable headers.

In [ ]:
# §5d — drill into ONE quote.
# Shortlist of the worst-predicted quotes is printed below (same data as the
# §5c table, sortable). Set (lemma, qidx) to any pair from it — defaults to the
# single hardest quote. Re-run this cell and the ones below after changing it.
worst = per_q.sort_values('difficulty', ascending=False)
print('Worst-predicted quotes (highest fraction of (provider×config) cells wrong):')
display(worst.head(15)[['quote_text', 'difficulty', 'n_correct', 'n_cells', 'gold_number']])

lemma, qidx = worst.index[0]            # default = hardest; override below if wanted
# lemma, qidx = 'PRIVILEGIUM', 833.0    # ← or paste a (lemma, quote_idx) here

# Resolve once; the cells below reuse ctx / preds / cand.
ctx, preds = quote_details(df_b, lemma, qidx, features_df=features)
cand = exp1_candidate_senses(lemma, qidx, gold_sid=ctx['gold_sid'])
print(f"\nselected: {lemma}  quote_idx={qidx}  (difficulty={worst.loc[(lemma, qidx), 'difficulty']:.2f})")
print(f"gold #{ctx['gold_number']}  {ctx['gold_sid']}  ({ctx.get('gold_label')})")
print(f"  la: {ctx.get('gold_def_la')}")
print(f"  pl: {ctx.get('gold_def_pl')}")
print()
print(ctx['quotation_wsd (LLM input)'])

In [ ]:
# Candidate-sense legend — the numbered menu the model saw for THIS quote
# (numbering is shuffled per quote; ★ = gold). Click a header to sort.
legend = cand.copy()
legend.insert(0, 'gold', legend.pop('is_gold').map({True: '★', False: ''}))
legend

In [ ]:
# Per-(provider, config) predictions, with the chosen sense's identity and
# glosses (pred_sid / def_la / def_pl). Click any header to sort.
preds

In [ ]:
# Provider × candidate-sense vote heatmap for this occurrence (gold column
# outlined, tick marked ★). Pass by='task_type' for prompt-style confusion.
plot_quote_confusion(df_b, lemma, qidx, gold_sid=ctx['gold_sid'])
plt.show()

In [ ]:
# Predicted-sense breakdown — count by provider (what the gold sense is
# confused with). One row per predicted sense; ★ = gold. Click a header to sort.
# Headline first: the single wrong sense predicted most often for this quote.
top = exp1_top_confusion(preds, cand)
if top.empty:
    print('No wrong predictions for this quote (every cell hit gold).')
else:
    t = top.iloc[0]
    print(f"Most-predicted WRONG sense: #{t['number']}  {t['label']}  ({t['sid']})")
    print(f"  {t['n_votes']} votes — {t['pct_of_wrong']}% of wrong picks, "
          f"{t['pct_of_all']}% of all picks")
    print(f"  la: {t['def_la']}")
    print(f"  pl: {t['def_pl']}")
    if len(top) > 1:
        runners = ', '.join(f"#{r.number} {r.label} ({r.n_votes})"
                            for r in top.iloc[1:].itertuples())
        print(f"  next: {runners}")
    print()
exp1_prediction_breakdown(preds, cand)

#### §5d.1. Cohort filter via `quotes_where`

`quotes_where(...)` returns a DataFrame; render it directly to scroll/sort,
or pass to `inspect_quotes(...)` for the print walk-through. Edit the
filter kwargs to investigate any subset.

In [ ]:
# Example: all sc-flagged quotes where ≥80% of cells got it wrong.
# Replace kwargs to investigate other cohorts (has_polish_gloss, genre, quotes_per_sense_bin, ...).
hard_sc = quotes_where(per_q, has_sc_note=True, difficulty_gte=0.8)
print(f'{len(hard_sc)} quotes match the filter; showing the table:')
quotes_table(hard_sc, top_n=20, sort_by='difficulty')

### §5e. Top sense confusion pairs

Most common (gold sense → predicted sense) confusions pooled across all
providers and configs. Reveals which sense distinctions models
systematically miss. Connects to §4's confusion matrices but at
quote-level granularity — useful for the writeup's "models confuse X
with Y" claims.

In [ ]:
# §5d — Top-K most common (gold, predicted) confusion pairs across all cells.
# Shows which sense distinctions models systematically can't make.
df_b['gold_int'] = pd.to_numeric(df_b['gold_number'], errors='coerce')
wrong = df_b[(df_b.outcome == 'wrong') & df_b['predicted'].notna() & df_b['gold_int'].notna()]
pairs = wrong.groupby(['lemma', 'gold_int', 'predicted']).size().reset_index(name='n')
pairs = pairs.sort_values('n', ascending=False).head(20)
print('Top 20 sense confusions (gold → predicted, pooled across providers/configs):')
print(pairs.to_string(index=False))

### §5f. Do the best models agree on hard quotes — and is it position bias?

Two questions about the worst-predicted quotes, both computable because within a
quote every model saw the **same shuffled candidate numbering** (seed = quote_idx,
def-language–independent): so `predicted` *is* the presentation position, comparing
votes across models within a quote is valid, and across quotes the shuffle decouples
position from sense (which identifies position bias).

- **§5f.1 agreement** — "best models" = top-K providers by accuracy; each provider
  is collapsed to its **modal vote** across its configs. Per hard quote: modal-share,
  distinct senses, entropy, and a consensus-correct / consensus-wrong / scatter
  class. Then inter-model κ vs κ-against-gold (do they agree with *each other* more
  than with the dictionary?) + a pairwise-κ heatmap; k-sensitivity at the end.
- **§5f.2 position bias** — for the senses the best models *agree* on but get wrong,
  is the agreed sense presented early? Normalized position vs the uniform null +
  P(position 1) vs chance + modal-share vs position. Front-loading ⇒ positional;
  uniform ⇒ the shared error is semantic.

In [ ]:
# §5f.1 — Best-model agreement on hard quotes (provider modal vote).
from itertools import combinations

K        = 4      # 'best models' = top-K providers by overall accuracy
DIFF_MIN = 0.8    # hard-quote threshold (fraction of cells wrong)
AGREE_MIN = 0.5   # modal-share needed to call a quote a 'consensus'

best = exp1_best_providers(df_b, k=K)
ncand = {lem: int(r['n_candidate_senses'])
         for lem, r in compute_lemma_features(df_b).iterrows()}
dfs       = exp1_attach_predicted_sid(df_b)          # adds predicted_sid (for κ)
votes_num = exp1_modal_votes(dfs, providers=best, value='predicted')      # numbers
votes_sid = exp1_modal_votes(dfs, providers=best, value='predicted_sid')  # for κ
gold_sid  = (dfs.dropna(subset=['quote_idx']).drop_duplicates(['lemma', 'quote_idx'])
             .set_index(['lemma', 'quote_idx'])['gold_sid'])

agg = exp1_agreement_metrics(votes_num, {i: ncand.get(i[0]) for i in votes_num.index})
agg = agg.join(per_q[['difficulty', 'gold_number', 'quote_text']])
agg['modal_is_gold'] = agg['modal'] == agg['gold_number']
agg['klass'] = np.where(agg['modal_share'] < AGREE_MIN, 'scatter',
                np.where(agg['modal_is_gold'], 'consensus-correct', 'consensus-wrong'))
hard = agg[agg['difficulty'] >= DIFF_MIN]

print(f"best-{K} providers by accuracy: {best}")
print(f"{len(hard)} hard quotes (difficulty ≥ {DIFF_MIN}); mean modal-share = {hard['modal_share'].mean():.2f}")
print('classification (hard):', hard['klass'].value_counts().to_dict())
print(f"→ on hard quotes the best-{K} reach a {AGREE_MIN:.0%}+ consensus on "
      f"{100*(hard['modal_share']>=AGREE_MIN).mean():.0f}% of quotes, and that agreed "
      f"sense is WRONG on {100*(hard['klass']=='consensus-wrong').mean():.0f}% of them.")

# Agree with EACH OTHER vs with GOLD? (hard cohort, on sense ids)
vh = votes_sid.loc[votes_sid.index.intersection(hard.index)]
kpair = {(a, b): kappa_labels(vh[a], vh[b]) for a, b in combinations(best, 2)}
kgold = {p: kappa_labels(vh[p], gold_sid.reindex(vh.index)) for p in best}
print(f"\nmean pairwise INTER-model κ = {np.nanmean(list(kpair.values())):.3f}   "
      f"vs   mean κ-vs-GOLD = {np.nanmean(list(kgold.values())):.3f}")
print("  (high inter-κ with near-zero gold-κ ⇒ a shared bias diverging from the dictionary)")

Kmat = pd.DataFrame(np.eye(len(best)), index=best, columns=best)
for (a, b), v in kpair.items():
    Kmat.loc[a, b] = Kmat.loc[b, a] = v
fig, ax = plt.subplots(figsize=(5.2, 4.2))
sns.heatmap(Kmat, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1,
            cbar_kws={'label': 'Cohen κ (sense ids)'}, ax=ax)
ax.set_title(f'Best-{K} inter-model agreement on hard quotes (κ)')
plt.tight_layout(); plt.show()

print('k-sensitivity (hard-quote consensus-wrong rate, mean modal-share):')
for k in (3, 4, 6):
    bp = exp1_best_providers(df_b, k=k)
    vn = exp1_modal_votes(dfs, providers=bp, value='predicted')
    a2 = exp1_agreement_metrics(vn, {i: ncand.get(i[0]) for i in vn.index}).join(per_q[['difficulty', 'gold_number']])
    h2 = a2[a2['difficulty'] >= DIFF_MIN]
    cw = 100*((h2['modal_share'] >= AGREE_MIN) & (h2['modal'] != h2['gold_number'])).mean()
    print(f"  k={k}: consensus-wrong={cw:.0f}%   mean modal-share={h2['modal_share'].mean():.2f}")

# κ by difficulty stratum — the non-circular check (cited in paper §6.3): do the
# models keep agreeing with EACH OTHER as their agreement with gold collapses?
print('\nκ by difficulty stratum (inter-model | vs-gold):')
for name, mask in [('easy ≤0.2', agg['difficulty'] <= 0.2),
                   ('mid 0.2–0.8', (agg['difficulty'] > 0.2) & (agg['difficulty'] < DIFF_MIN)),
                   (f'hard ≥{DIFF_MIN}', agg['difficulty'] >= DIFF_MIN)]:
    v = votes_sid.loc[votes_sid.index.intersection(agg[mask].index)]
    ki = np.nanmean([kappa_labels(v[a], v[b]) for a, b in combinations(best, 2)])
    kg = np.nanmean([kappa_labels(v[p], gold_sid.reindex(v.index)) for p in best])
    print(f"  {name:12} n={len(v):3}  inter-model κ={ki:.2f}   κ-vs-gold={kg:.2f}")

In [ ]:
# §5f.1 (cont.) — The systematic shared failures: best-K agree on a WRONG sense.
# Sortable; this is where the model consensus and the dictionary disagree.
cw_quotes = hard[hard['klass'] == 'consensus-wrong'].sort_values(
    ['difficulty', 'modal_share'], ascending=False)
rows = []
for (lemma, qidx), r in cw_quotes.head(25).iterrows():
    cand = exp1_candidate_senses(lemma, qidx, gold_sid=gold_sid.get((lemma, qidx)))
    m, g = int(r['modal']), int(r['gold_number'])
    rows.append({
        'lemma': lemma, 'quote_idx': qidx,
        'diff': round(r['difficulty'], 2), 'agree': round(r['modal_share'], 2),
        'agreed_wrong': f"#{m} {cand.loc[m, 'label'] if m in cand.index else '?'}",
        'agreed_gloss_la': cand.loc[m, 'def_la'] if m in cand.index else '',
        'gold': f"#{g} {cand.loc[g, 'label'] if g in cand.index else '?'}",
        'gold_gloss_la': cand.loc[g, 'def_la'] if g in cand.index else '',
    })
pd.DataFrame(rows)

In [ ]:
# §5f.2 — Is the agreed WRONG pick just presentation position (primacy)?
# position = predicted number; normalized by #candidates. Decoupled from sense
# across quotes by the per-quote shuffle, so a top-of-list lean is real bias.
cw = hard[hard['klass'] == 'consensus-wrong'].copy()
cw['ncand'] = [ncand.get(i[0]) for i in cw.index]
cw['pos'] = cw['modal'].astype(int)
cw['pos_norm'] = (cw['pos'] - 1) / (cw['ncand'] - 1)

mean_pn = cw['pos_norm'].mean()
rng = np.random.default_rng(RNG_SEED)
vals = cw['pos_norm'].to_numpy()
boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(2000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
p1_obs = (cw['pos'] == 1).mean()
p1_exp = np.mean([0 if gp == 1 else 1/(nc-1) for gp, nc in zip(cw['gold_number'], cw['ncand'])])
modal_share = hard['modal_share'].mean()

print(f"agreed wrong sense — mean normalized position = {mean_pn:.3f}  "
      f"[95% CI {lo:.3f}, {hi:.3f}]   (uniform null = 0.5)")
print(f"P(agreed wrong at position 1)               = {p1_obs:.3f}   (chance ≈ {p1_exp:.3f})")

# Two separate signals: an overall front/back lean (mean vs 0.5) and a
# first-slot grab (P(pos 1) vs chance) — report both, with magnitude.
if hi < 0.5:
    lean = f"mild FRONT-loading (mean {mean_pn:.2f} < 0.5, CI excludes 0.5; effect ≈ {0.5-mean_pn:+.2f} on [0,1])"
elif lo > 0.5:
    lean = f"back-loading (mean {mean_pn:.2f} > 0.5)"
else:
    lean = f"no positional lean (CI {lo:.2f}–{hi:.2f} includes 0.5)"
slot1 = ("at chance" if abs(p1_obs - p1_exp) < 0.03
         else ("ABOVE chance" if p1_obs > p1_exp else "below chance"))
print(f"→ overall lean: {lean}")
print(f"→ first slot  : P(pos 1) is {slot1} — {'primacy grab' if slot1=='ABOVE chance' else 'no first-slot grab'}")
print(f"→ but modal-share is {modal_share:.2f}; a {0.5-mean_pn:+.2f} positional lean is far too small to "
      f"produce that consensus ⇒ the shared error is predominantly SEMANTIC, with at most a minor positional component.")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(cw['pos_norm'], bins=10, range=(0, 1), color='C0', alpha=.85)
axes[0].axvline(0.5, color='k', ls='--', label='uniform mean (0.5)')
axes[0].axvline(mean_pn, color='crimson', label=f'observed ({mean_pn:.2f})')
axes[0].set_xlabel('normalized presentation position of agreed wrong sense')
axes[0].set_ylabel('# consensus-wrong quotes'); axes[0].legend()
byp = hard.assign(pos=hard['modal'].astype(int)).groupby('pos')['modal_share'].mean()
axes[1].bar(byp.index, byp.values, color='C2')
axes[1].set_xlabel('agreed sense presentation position')
axes[1].set_ylabel('mean modal-share'); axes[1].set_title('does consensus rise when shown early?')
plt.tight_layout(); plt.show()

### §5g. When wrong, *how* wrong? Dictionary-hierarchy distance

A graded "how bad is the error" view: when a model misses, does it land on a
**sibling** sense (same major sense, e.g. I.1 vs I.2) or jump to a **different
major sense** (I vs II)? The hierarchy is read from the **label path**
(`II.A.2.a` → `('II','A','2','a')`) — the dictionary's authoritative nesting —
not the sid integers (which are a renumbered internal id: e.g. `bellum.2` and
`bellum.3` are both labelled *II*).

Every number is compared to a **random-wrong baseline** — the relationship you'd
expect by picking uniformly among that quote's *non-gold* candidates — so the
rates account for each lemma's tree shape (a 3-sense lemma affords fewer
"siblings" than a 16-sense one). Note: the model never *saw* the hierarchy (the
prompt is a flat numbered list), so this measures whether errors are
*semantically adjacent* in the dictionary, not whether the model used the tree.

- **§5g.1** — observed vs baseline distance over all wrong picks.
- **§5g.2** — by provider (do stronger models err closer?) + the distance of the
  §5f *consensus-wrong* agreed senses.

In [ ]:
# §5g.1 — Hierarchy distance of WRONG picks vs a random-wrong baseline.
s2l = exp1_label_map()
dfs = exp1_attach_predicted_sid(df_b)
wrong = dfs[(dfs.outcome == 'wrong') & dfs.predicted_sid.notna()
            & dfs.gold_sid.notna()].copy()
rel = wrong.apply(lambda r: sid_hierarchy_relation(r.gold_sid, r.predicted_sid, s2l),
                  axis=1, result_type='expand')
wrong = pd.concat([wrong, rel], axis=1)

CATS = ['sibling', 'parent-child', 'same-major-distant', 'different-major', 'same-label']
gold_sid = (dfs.dropna(subset=['quote_idx']).drop_duplicates(['lemma', 'quote_idx'])
            .set_index(['lemma', 'quote_idx'])['gold_sid'])
brows = {}
for (lemma, qidx), gsid in gold_sid.items():
    cand = exp1_candidate_senses(lemma, qidx, gold_sid=gsid)
    b = pd.DataFrame([sid_hierarchy_relation(gsid, sid, s2l)
                      for sid in cand['sid'] if sid != gsid])
    if b.empty:
        continue
    d = {f'b_{c}': float((b.relation == c).mean()) for c in CATS}
    d['b_same_major'] = float(b.same_major.mean()); d['b_tree'] = float(b.tree_dist.mean())
    brows[(lemma, qidx)] = d
base = pd.DataFrame(brows).T
wb = base.reindex(wrong.set_index(['lemma', 'quote_idx']).index)   # per-pick baseline

obs = {c: (wrong.relation == c).mean() for c in CATS}
print('WRONG-pick hierarchy distance — observed vs random-wrong baseline:')
print(f"  same major sense  : {100*wrong.same_major.mean():5.1f}%   (chance {100*wb.b_same_major.mean():5.1f}%)")
print(f"  sibling (I.1↔I.2) : {100*obs['sibling']:5.1f}%   (chance {100*wb['b_sibling'].mean():5.1f}%)")
print(f"  parent / child    : {100*obs['parent-child']:5.1f}%   (chance {100*wb['b_parent-child'].mean():5.1f}%)")
print(f"  different major   : {100*obs['different-major']:5.1f}%   (chance {100*wb['b_different-major'].mean():5.1f}%)")
print(f"  mean tree dist    : {wrong.tree_dist.mean():5.2f}    (chance {wb.b_tree.mean():5.2f})")
d_tree = wb.b_tree.mean() - wrong.tree_dist.mean()
print(f"→ errors are {d_tree:+.2f} edges closer than random selection among candidates "
      f"({'modestly' if d_tree < 0.8 else 'clearly'} better than chance); "
      f"but {100*obs['different-major']:.0f}% still jump to a DIFFERENT major sense.")

fig, ax = plt.subplots(figsize=(7.5, 4))
order = ['sibling', 'parent-child', 'same-major-distant', 'different-major']
x = np.arange(len(order)); w = 0.38
ax.bar(x - w/2, [100*obs[c] for c in order], w, label='observed (wrong picks)', color='C3')
ax.bar(x + w/2, [100*wb[f'b_{c}'].mean() for c in order], w, label='random-wrong baseline', color='0.7')
ax.set_xticks(x); ax.set_xticklabels(order, rotation=15)
ax.set_ylabel('% of wrong picks'); ax.legend()
ax.set_title('How wrong? gold→predicted hierarchy relation vs chance')
plt.tight_layout(); plt.show()

In [ ]:
# §5g.2 — Do stronger models err CLOSER? + the §5f consensus-wrong distances.
prov = (wrong.groupby('provider_label')
        .agg(n_wrong=('relation', 'size'),
             same_major=('same_major', 'mean'),
             mean_tree=('tree_dist', 'mean'),
             sibling=('relation', lambda s: (s == 'sibling').mean()),
             diff_major=('relation', lambda s: (s == 'different-major').mean())))
prov.insert(0, 'accuracy', df_b.groupby('provider_label').success.mean())
prov = prov.sort_values('accuracy', ascending=False)
print('Per-provider error closeness (sorted by accuracy):')
display(prov[['accuracy', 'same_major', 'sibling', 'diff_major', 'mean_tree', 'n_wrong']].round(3))
r = prov['accuracy'].corr(prov['mean_tree'])
print(f"corr(accuracy, mean tree-dist of errors) = {r:+.2f}  "
      f"→ {'stronger models err closer' if r < -0.3 else 'no clear closer-error trend with accuracy'}")

# §5f tie-in: are the senses the best models AGREE on (consensus-wrong) near or gross?
cw = hard[hard['klass'] == 'consensus-wrong']
crows = []
for (lemma, qidx), rrow in cw.iterrows():
    cand = exp1_candidate_senses(lemma, qidx, gold_sid=gold_sid.get((lemma, qidx)))
    m = int(rrow['modal'])
    if m in cand.index:
        crows.append(sid_hierarchy_relation(gold_sid[(lemma, qidx)], cand.loc[m, 'sid'], s2l))
cw_rel = pd.DataFrame(crows)
print(f"\n§5f consensus-wrong agreed sense vs gold ({len(cw_rel)} quotes):")
print((100*cw_rel.relation.value_counts(normalize=True)).round(1).to_string())
print(f"  same-major {100*cw_rel.same_major.mean():.0f}%   mean tree {cw_rel.tree_dist.mean():.2f}  "
      f"(vs all-wrong same-major {100*wrong.same_major.mean():.0f}%, tree {wrong.tree_dist.mean():.2f})")

## §6. Phase A → Phase B replication panel

The *generalization claim*. For each (provider, template, def_language) cell
common to both phases: Phase A acc vs Phase B acc. Two-sample (not paired —
the lemma sets are disjoint by design). Holm-corrected across the family
of cell comparisons.

Headline outputs:
- Spearman ρ on per-provider rankings (Phase A → Phase B)
- Per-template ranking stability
- Does pl > la replicate?
- Scatter plot of Phase A acc × Phase B acc per cell

In [ ]:
if df_a is None:
    print(f'PHASE_A is None — skipping replication panel. Set PHASE_A at top to enable.')
else:
    # Filter Phase A to the same axis subset as Phase B (gran=l1, meta=F, plain only)
    df_a_match = df_a[(df_a.granularity == 'l1') & (df_a.show_metadata == False)
                      & (df_a.yaml_kind == 'plain')].copy()
    df_a_match['config'] = df_a_match.apply(
        lambda r: f"{r.template}|def={r.def_language}", axis=1)

    a_cells = df_a_match.groupby(['provider_label','config']).agg(
        n=('success','size'), acc_A=('success','mean')).reset_index()
    b_cells = df_b.groupby(['provider_label','config']).agg(
        n=('success','size'), acc_B=('success','mean')).reset_index()
    rep = a_cells.merge(b_cells, on=['provider_label','config'],
                        suffixes=('_A', '_B'))
    rep['delta_pp'] = (rep['acc_B'] - rep['acc_A']) * 100
    rep['acc_A%'] = (rep['acc_A'] * 100).round(2)
    rep['acc_B%'] = (rep['acc_B'] * 100).round(2)
    print(f'Cells common to both phases: {len(rep)}')
    rep[['provider_label','config','n_A','n_B','acc_A%','acc_B%','delta_pp']].sort_values('delta_pp')

In [ ]:
if df_a is not None:
    from scipy.stats import spearmanr
    # Per-provider rankings (averaged across configs) — does the provider order hold?
    prov_a = df_a_match.groupby('provider_label')['success'].mean()
    prov_b = df_b.groupby('provider_label')['success'].mean()
    common_p = sorted(set(prov_a.index) & set(prov_b.index))
    rho, p = spearmanr([prov_a[p] for p in common_p], [prov_b[p] for p in common_p])
    print(f'Per-provider ranking Spearman ρ (Phase A → Phase B): {rho:.3f}  (p={p:.4f}, n={len(common_p)})')
    print(f'  → {rho:.0%} rank correlation across phases — generalization {"holds" if rho > 0.7 else "is partial" if rho > 0.4 else "is weak"}')

    # Per-template ranking
    tpl_a = df_a_match.groupby('template')['success'].mean()
    tpl_b = df_b.groupby('template')['success'].mean()
    common_t = sorted(set(tpl_a.index) & set(tpl_b.index))
    if len(common_t) >= 3:
        rho_t, p_t = spearmanr([tpl_a[t] for t in common_t], [tpl_b[t] for t in common_t])
        print(f'\nPer-template ranking Spearman ρ: {rho_t:.3f}  (p={p_t:.4f}, n={len(common_t)})')

    # pl vs la replication
    for phase, dfx in [('A', df_a_match), ('B', df_b)]:
        la = dfx[dfx.def_language == 'la']['success'].mean()
        pl = dfx[dfx.def_language == 'pl']['success'].mean()
        print(f'Phase {phase}: la={100*la:.2f}% vs pl={100*pl:.2f}% → Δ={100*(pl-la):+.2f}pp')

In [ ]:
if df_a is not None and not rep.empty:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(rep['acc_A%'], rep['acc_B%'], alpha=0.6)
    lim = max(rep['acc_A%'].max(), rep['acc_B%'].max()) + 5
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='y = x (perfect replication)')
    ax.set_xlabel('Phase A accuracy %'); ax.set_ylabel('Phase B accuracy %')
    ax.set_title('Per (provider, config) cell — Phase A → Phase B')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## §7. Pooled descriptive analyses (38 lemmas)

Things that are *not* selection-conditional. Pool Phase A + Phase B on the
axes that the CI-overlap rule didn't touch (template, def_language, provider).
Strictly descriptive — no claims about "best model" come from here.

In [ ]:
if df_a is None:
    print('PHASE_A is None — skipping pooled section.')
else:
    df_pool = pd.concat([df_a_match.assign(phase='A'), df_b.assign(phase='B')], ignore_index=True)
    print(f'Pooled corpus: {df_pool.lemma.nunique()} lemmas, {len(df_pool):,} rows')
    print(f'  Phase A lemmas: {df_a_match.lemma.nunique()} | Phase B lemmas: {df_b.lemma.nunique()}')

In [ ]:
if df_a is not None:
    # Cross-lemma stability heatmap across 38 lemmas
    heat = (df_pool.groupby(['provider_label', 'lemma'])['success'].mean() * 100).unstack()
    heat = heat.reindex(index=heat.mean(axis=1).sort_values(ascending=False).index)
    fig, ax = plt.subplots(figsize=(max(10, 0.35 * heat.shape[1]), 0.4 * heat.shape[0] + 1))
    sns.heatmap(heat, annot=True, fmt='.0f', cmap='YlGnBu', ax=ax,
                cbar_kws={'label': 'acc %'})
    ax.set_title('Per-lemma × provider accuracy — pooled Phase A + B (38 lemmas)')
    plt.tight_layout(); plt.show()

In [ ]:
if df_a is not None:
    # Per-lemma difficulty distribution — ranked
    per_lemma = (df_pool.groupby('lemma')['success'].mean() * 100).sort_values()
    fig, ax = plt.subplots(figsize=(9, max(4, 0.25 * len(per_lemma))))
    colors = ['C2' if l not in PHASE_A_LEMMAS else 'C0' for l in per_lemma.index]
    ax.barh(range(len(per_lemma)), per_lemma.values, color=colors)
    ax.set_yticks(range(len(per_lemma))); ax.set_yticklabels(per_lemma.index, fontsize=8)
    ax.set_xlabel('Mean accuracy % across providers')
    ax.set_title('Per-lemma difficulty (blue=Phase A, green=Phase B)')
    ax.axvline(per_lemma.mean(), color='black', linestyle='--', alpha=0.5,
               label=f'mean = {per_lemma.mean():.1f}%')
    ax.legend(); plt.tight_layout(); plt.show()

## §8. Dev-set appendix table — *Phase A on the same configs*

Same headline metrics as §2, computed on Phase A lemmas. Explicit
selection-bias disclaimer. Becomes an appendix table in the paper.

In [ ]:
if df_a is None:
    print('PHASE_A is None — skipping dev appendix.')
else:
    df_a_match['predicted'] = df_a_match['output'].map(parse_predicted_number)
    mfs_macro_phaseA = float(np.mean([mfs_per_lemma[l] for l in df_a_match.lemma.unique() if l in mfs_per_lemma]))
    dev = headline_table(df_a_match, mfs_macro_phaseA)
    print(f'Phase A dev-appendix table (MFS macro = {100*mfs_macro_phaseA:.2f}%):')
    print('  ⚠ Configs were SELECTED on this data via CI-overlap (entry 2026-05-16c).')
    print('  ⚠ Numbers inherit selection-favorable bias. Not headline.')
    dev

## §9. Methodology notes for the paper

Mirrors `docs/methodology_log.md` style. Edit to capture what this Phase B
run actually concluded — this is the local-to-notebook version that travels
with the analysis.

**Supports (paper claims):**
- §2 — per-(provider, config) headline accuracy + κ + MFS-comparison on disjoint Phase B lemmas
- §3 — MFS baseline + flag for sub-MFS configs ("not a finding" per PLAN.md §2.5)
- §4 — per-sense P/R/F1, macro F1, confusion matrices
- §6 — Phase A → Phase B generalization (per-provider + per-template Spearman ρ, def_language replication)

**Supports descriptively (no "best model" claims from these):**
- §5 — quote-level error patterns, covariate slices (Polish gloss, genre, …)
- §7 — pooled 38-lemma analyses on non-selection-conditional axes

**Does NOT support — defer to v2 / separate evals:**
- 0-assignment analysis (PLAN.md §4.1)
- No-context baseline (needs PHASE_B_NO_CONTEXT eval)
- Latin BERT baseline (separate pipeline)
- Contamination split (needs probe data — PLAN.md §2.6)

**Pre-registered analysis plan**: §2 headline + §3 MFS + §6 replication form the
pre-registered analyses. §5 is exploratory and must be labelled as such in the writeup.

**Open follow-ups (noted from §5b analysis, deferred — not in scope for v1):**
- **Polish-gloss confound test** — the +4.6 pp `has_polish_gloss × def=pl` effect could be real disambiguation help OR surface-string matching against `def_pl` (gloss word literally appears in the gold sense's Polish definition). Test by extracting the Polish substring inside `"…"` / `„…"` and checking overlap with the gold's `def_pl`. Split `has_polish_gloss=True` accordingly; if the benefit collapses in the no-overlap subset, the confound dominates.
- **Sc-note "lack of context" mechanism** — the −12.3 pp on sc-flagged quotes is consistent across all templates (§5b.4), so it's not a template-recoverable issue. Two unconfirmed sub-mechanisms: (a) sc-quotes may be systematically shorter (n_words distribution check); (b) sc-content type (referent vs entity vs implied verb) may correlate with failure category. Sub-mechanism (a) is a one-liner; (b) needs ~15 min of hand-categorisation via `inspect_quotes(per_q[per_q.has_sc_note==True], df_b, max_n=10)`.
